# FAISS

Meta에서 개발한 벡터 유사도 검색 라이브러리 입니다.

CPU, GPU


In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_ollama.embeddings import OllamaEmbeddings

import json

text_loader = TextLoader(
    file_path='./libs/qna_data.json',
    encoding='utf-8',
)

text_docs = text_loader.load()
json_docs = json.loads(text_docs[0].page_content)

docs = [
    Document(
        page_content=f"질문: {doc['question']}\n\n답변: {doc['answer']}",
        metadata={
            'id': doc['id'],
            'category': doc['category'],
            'keywords': doc['keywords'],
        }
    )
    for doc in json_docs
]

embedding = OllamaEmbeddings(model='qwen3-embedding:latest')

vector_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding
)

print(f'입력 문서 수 : {len(docs)}')
print(f'저장된 벡터 수 : {vector_store.index.ntotal}')
print(f'벡터 차원 : {vector_store.index.d}')

입력 문서 수 : 10
저장된 벡터 수 : 10
벡터 차원 : 4096


In [6]:
vector_store.save_local('faiss_index')

In [8]:
load_vectorstore = FAISS.load_local(
    'faiss_index',
    embeddings=embedding,
    allow_dangerous_deserialization=True,
)


In [9]:
user_query = '제품이 마음에안들어요 취소나 환불 가능할까요?'

results = load_vectorstore.similarity_search(
    query=user_query,
    k=3
)

for result in results:
    print(result)


page_content='질문: 주문을 취소하고 싶은데 어떻게 하나요?

답변: 상품 발송 전까지는 마이페이지 > 주문내역에서 직접 취소하실 수 있습니다. 이미 배송이 시작된 경우에는 상품 수령 후 반품 절차를 진행하셔야 합니다. 무통장 입금으로 결제하신 경우 입금 전이라면 자동 취소되며, 입금 후에는 2-3일 내 환불 처리됩니다. 카드 결제는 취소 승인 후 카드사에 따라 3-7일 소요됩니다.' metadata={'id': 'qna_009', 'category': '주문취소', 'keywords': ['주문취소', '취소', '환불', '취소방법', '결제취소']}
page_content='질문: 사이즈가 맞지 않는데 교환할 수 있나요?

답변: 네, 상품 수령 후 7일 이내 교환 신청이 가능합니다. 마이페이지에서 교환 신청 후 원하시는 사이즈를 선택해 주세요. 교환 상품의 재고가 있는 경우 회수 후 3-5일 이내에 새 상품이 발송됩니다. 단, 고객 변심으로 인한 교환 시 왕복 배송비가 발생하며, 제품 하자인 경우 무료로 교환해 드립니다.' metadata={'id': 'qna_006', 'category': '교환', 'keywords': ['교환', '사이즈교환', '교환신청', '교환기간', '교환배송비']}
page_content='질문: 상품 반품은 어떻게 하나요?

답변: 상품 수령 후 7일 이내에 반품 신청이 가능합니다. 마이페이지 > 주문내역에서 반품 신청을 하시면 됩니다. 단, 상품 택과 라벨이 훼손되지 않은 미사용 제품에 한하며, 식품이나 개봉한 화장품 등 일부 상품은 반품이 제한될 수 있습니다. 반품 배송비는 고객 변심의 경우 왕복 배송비 6,000원이 차감됩니다.' metadata={'id': 'qna_002', 'category': '반품', 'keywords': ['반품', '반품신청', '반품기간', '반품조건', '배송비']}
